# nano-prover — fixed Kaggle runner

**One-time setup** (do once per Kaggle notebook):
1. Right panel → **Settings → Accelerator → `GPU T4 x2`**
2. Right panel → **Settings → Internet → On**
3. Top menu → **Add-ons → Secrets** → add a secret named **`GITHUB_TOKEN`** = your GitHub token
   (github.com → Settings → Developer settings → Personal access tokens → Tokens (classic) → scope `repo`).

**Every run after that:** just press ▶ on the cell below. It pulls the latest repo, trains with the
config in `training/scripts/run_kaggle.sh`, and pushes back **both** the generated `preds.jsonl` and
the **trained adapter** `training/model/` (your deliverable model). To change the next run, edit
`run_kaggle.sh` in the repo and `git push` — you never edit this notebook. When it finishes, locally
run `git pull` (you now have the model in `training/model/`) and `python -m assess`.

In [ ]:
# ---- FIXED: pull latest repo → train (config from run_kaggle.sh) → push adapter + preds. Do not edit. ----
import os
os.chdir('/kaggle/working')
!rm -rf Conjecture-Gym && git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/kaggle/working/Conjecture-Gym')
!nvidia-smi -L
!bash training/scripts/run_kaggle.sh

# push the trained adapter (training/model/) + generated preds back, via the Kaggle Secret 'GITHUB_TOKEN'
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
!git add -f training/model training/scripts/preds.jsonl
!git -c user.email=kaggle@run.local -c user.name=kaggle commit -m 'trained adapter + preds from kaggle' || true
!git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main
print('\nDONE. Locally:  git pull  (model is now in training/model/)  &&  python -m assess')